Get max date for wheater data

In [3]:
import pandas as pd

path = "../data/interim/taxi_2026_xsv_cleaned.csv"

df = pd.read_csv(
    path,
    dtype={"Trip Start Timestamp": "string"},
    usecols=["Trip Start Timestamp"]
)

df["trip_start_dt"] = pd.to_datetime(
    df["Trip Start Timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

print("Min:", df["trip_start_dt"].min())
print("Max:", df["trip_start_dt"].max())
print("Missing parsed timestamps:", df["trip_start_dt"].isna().sum())

Min: 2026-01-01 00:00:00
Max: 2026-05-01 00:00:00
Missing parsed timestamps: 0


check weather data

In [6]:
import pandas as pd

path = "../data/weather_2024_NCEI_OHare.csv"

weather_sample = pd.read_csv(path, nrows=5)
print(weather_sample.columns.tolist())
print(weather_sample.head())

['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'SOURCE', 'HourlyAltimeterSetting', 'HourlyDewPointTemperature', 'HourlyDryBulbTemperature', 'HourlyPrecipitation', 'HourlyPresentWeatherType', 'HourlyPressureChange', 'HourlyPressureTendency', 'HourlyRelativeHumidity', 'HourlySkyConditions', 'HourlySeaLevelPressure', 'HourlyStationPressure', 'HourlyVisibility', 'HourlyWetBulbTemperature', 'HourlyWindDirection', 'HourlyWindGustSpeed', 'HourlyWindSpeed', 'Sunrise', 'Sunset', 'DailyAverageDewPointTemperature', 'DailyAverageDryBulbTemperature', 'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure', 'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature', 'DailyAverageWindSpeed', 'DailyCoolingDegreeDays', 'DailyDepartureFromNormalAverageTemperature', 'DailyHeatingDegreeDays', 'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature', 'DailyPeakWindDirection', 'DailyPeakWindSpeed', 'DailyPrecipitation', 'DailySnowDepth', 'DailySnowfall

In [29]:
from pathlib import Path
import pandas as pd

input_path = "../data/weather_2024_NCEI_OHare.csv"
output_path = "../data/processed/weather_2024_NCEI_OHare_hourly.csv"
paths = "../"

Path("../data/processed").mkdir(parents=True, exist_ok=True)

weather = pd.read_csv(input_path, dtype="string", low_memory=False)

weather["DATE"] = pd.to_datetime(weather["DATE"], errors="coerce")

weather_cols = [
    "STATION",
    "DATE",
    "LATITUDE",
    "LONGITUDE",
    "NAME",
    "REPORT_TYPE",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyRelativeHumidity",
    "HourlyPrecipitation",
    "HourlyWindSpeed",
    "HourlyVisibility",
    "HourlyPresentWeatherType",
    "HourlySkyConditions",
    "HourlySeaLevelPressure",
]

weather = weather[[col for col in weather_cols if col in weather.columns]].copy()

In [30]:
def clean_noaa_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("T", "0", regex=False)
        .str.replace("s", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

weather["temperature"] = clean_noaa_numeric(weather["HourlyDryBulbTemperature"])
weather["dew_point"] = clean_noaa_numeric(weather["HourlyDewPointTemperature"])
weather["humidity"] = clean_noaa_numeric(weather["HourlyRelativeHumidity"])
weather["precipitation"] = clean_noaa_numeric(weather["HourlyPrecipitation"])
weather["wind_speed"] = clean_noaa_numeric(weather["HourlyWindSpeed"])
weather["visibility"] = clean_noaa_numeric(weather["HourlyVisibility"])
weather["sea_level_pressure"] = clean_noaa_numeric(weather["HourlySeaLevelPressure"])

aggregate weather data to hourly level and save as parquet file for further processing.

In [31]:
weather["time_bucket_hour"] = weather["DATE"].dt.floor("h")

weather_hourly = (
    weather
    .groupby("time_bucket_hour", as_index=False)
    .agg({
        "temperature": "mean",
        "dew_point": "mean",
        "humidity": "mean",
        "precipitation": "sum",
        "wind_speed": "mean",
        "visibility": "mean",
        "sea_level_pressure": "mean",
        "HourlyPresentWeatherType": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA,
        "HourlySkyConditions": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA,
    })
)

fix missing values

In [32]:
weather_hourly["precipitation"] = weather_hourly["precipitation"].fillna(0)

for col in ["temperature", "dew_point", "humidity", "wind_speed", "visibility", "sea_level_pressure"]:
    weather_hourly[col] = weather_hourly[col].ffill().bfill()

In [33]:
weather_hourly.to_csv(output_path, index=False)

print(weather_hourly.head())
print(weather_hourly.tail())
print(weather_hourly.isna().mean().sort_values(ascending=False))
print("Min:", weather_hourly["time_bucket_hour"].min())
print("Max:", weather_hourly["time_bucket_hour"].max())
print("Rows:", len(weather_hourly))

     time_bucket_hour  temperature  dew_point  humidity  precipitation  \
0 2024-01-01 00:00:00         32.0       26.5      80.5            0.0   
1 2024-01-01 01:00:00         32.0       26.0      79.0            0.0   
2 2024-01-01 02:00:00         32.0       26.0      79.0            0.0   
3 2024-01-01 03:00:00    31.666667       26.0      80.0            0.0   
4 2024-01-01 04:00:00         31.0       24.0      76.0            0.0   

   wind_speed  visibility  sea_level_pressure HourlyPresentWeatherType  \
0        14.0        9.97              30.155                     <NA>   
1        16.0         9.0               30.18             -SN:03 |SN |   
2   12.666667    9.666667                30.2                     <NA>   
3   12.333333        9.98              30.205                     <NA>   
4   14.333333        10.0               30.23                     <NA>   

   HourlySkyConditions  
0                   15  
1  BKN:07 18 OVC:08 55  
2  BKN:07 18 OVC:08 55  
3         

Some rows are missing. The year 2024 has 8784 hours (366 days * 24 hours) but we only have 8783 rows.

In [34]:
import pandas as pd

# Make sure time column is datetime
weather_hourly["time_bucket_hour"] = pd.to_datetime(weather_hourly["time_bucket_hour"])

# Create complete hourly range for 2024
full_hours = pd.DataFrame({
    "time_bucket_hour": pd.date_range(
        start="2024-01-01 00:00:00",
        end="2024-12-31 23:00:00",
        freq="h"
    )
})

# Left join weather onto complete hourly index
weather_hourly_complete = full_hours.merge(
    weather_hourly,
    on="time_bucket_hour",
    how="left"
)

print("Rows before:", len(weather_hourly))
print("Rows after:", len(weather_hourly_complete))
print(weather_hourly_complete.isna().mean().sort_values(ascending=False))

Rows before: 8768
Rows after: 8784
HourlyPresentWeatherType    0.869649
HourlySkyConditions         0.064777
temperature                 0.001821
dew_point                   0.001821
humidity                    0.001821
precipitation               0.001821
wind_speed                  0.001821
visibility                  0.001821
sea_level_pressure          0.001821
time_bucket_hour            0.000000
dtype: float64


In [35]:
# Precipitation: missing means no recorded precipitation
weather_hourly_complete["precipitation"] = weather_hourly_complete["precipitation"].fillna(0)

# Continuous weather variables: forward/backward fill
continuous_cols = [
    "temperature",
    "dew_point",
    "humidity",
    "wind_speed",
    "visibility",
    "sea_level_pressure"
]

for col in continuous_cols:
    weather_hourly_complete[col] = weather_hourly_complete[col].ffill().bfill()

# Text columns: keep missing as "None"
weather_hourly_complete["HourlyPresentWeatherType"] = (
    weather_hourly_complete["HourlyPresentWeatherType"].fillna("None")
)

weather_hourly_complete["HourlySkyConditions"] = (
    weather_hourly_complete["HourlySkyConditions"].fillna("Unknown")
)

add new features

In [41]:
weather_hourly_complete["has_precipitation"] = (
    weather_hourly_complete["precipitation"] > 0
).astype(int)

weather_hourly_complete["has_snow"] = (
    weather_hourly_complete["HourlyPresentWeatherType"]
    .astype(str)
    .str.contains("SN", case=False, na=False)
).astype(int)

weather_hourly_complete["has_rain"] = (
    weather_hourly_complete["HourlyPresentWeatherType"]
    .astype(str)
    .str.contains("RA", case=False, na=False)
).astype(int)

weather_hourly_complete["low_visibility"] = (
    weather_hourly_complete["visibility"] < 5
).astype(int)

weather_hourly_complete["strong_wind"] = (
    weather_hourly_complete["wind_speed"] >= 20
).astype(int)

weather_hourly_complete["is_overcast"] = weather_hourly_complete["HourlySkyConditions"].astype(str).str.contains("OVC", na=False).astype(int)
weather_hourly_complete["is_cloudy"] = weather_hourly_complete["HourlySkyConditions"].astype(str).str.contains("BKN|OVC", na=False).astype(int)

use celsius insted of fahrenheit

In [42]:
weather_hourly_complete["temperature_c"] = (
    weather_hourly_complete["temperature"] - 32
) * 5 / 9

weather_hourly_complete["dew_point_c"] = (
    weather_hourly_complete["dew_point"] - 32
) * 5 / 9

In [43]:
weather_hourly_complete.to_csv(
    "data/processed/weather_ohare_2024_hourly.csv",
    index=False
)

In [44]:
weather_hourly_complete.head()

,time_bucket_hour,temperature,dew_point,humidity,precipitation,wind_speed,visibility,sea_level_pressure,HourlyPresentWeatherType,HourlySkyConditions,has_precipitation,has_snow,has_rain,low_visibility,strong_wind,temperature_c,dew_point_c,is_overcast,is_cloudy
0,2024-01-01 00:00:00,32.0,26.5,80.5,0.0,14.0,9.97,30.155,None,15,0,0,0,0,0,0.0,-3.055556,0,0
1,2024-01-01 01:00:00,32.0,26.0,79.0,0.0,16.0,9.0,30.18,-SN:03 |SN |,BKN:07 18 OVC:08 55,0,1,0,0,0,0.0,-3.333333,1,1
2,2024-01-01 02:00:00,32.0,26.0,79.0,0.0,12.666667,9.666667,30.2,None,BKN:07 18 OVC:08 55,0,0,0,0,0,0.0,-3.333333,1,1
3,2024-01-01 03:00:00,31.666667,26.0,80.0,0.0,12.333333,9.98,30.205,None,15,0,0,0,0,0,-0.185185,-3.333333,0,0
4,2024-01-01 04:00:00,31.0,24.0,76.0,0.0,14.333333,10.0,30.23,None,SCT:04 19 OVC:08 44,0,0,0,0,0,-0.555556,-4.444444,1,1
